In [1]:
!pip -q install -U langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.9 MB/s eta 0:00:00


In [ ]:
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]=input("Enter your groq api key:")
print("langchain + groq setup complete!")

In [4]:
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
print("llm created!")

llm created!


In [5]:
response=llm.invoke("what is an ai agent? explain in one sentence")
print(response.content)

An AI agent is a software entity that perceives its environment through data inputs, processes that information using algorithms (often machine learning), and takes autonomous actions to achieve specific goals.


In [7]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_messages([
    ("system","you are helpful ai tutor"),
    ("human","{question}")
]

)
chain = prompt | llm
response=chain.invoke({
    "question":"explain ai agents in one simple sentence."
    })
print(response.content)

An AI agent is a software system that perceives its environment, makes decisions, and takes actions autonomously to achieve a goal.


In [8]:
response=llm.invoke("""
You are an AI agent.
User:What is 25*4?
decide what action you should take to answer the user.
respond with only:
ACTION:<your action
""")
print(response.content)

ACTION:ANSWER 100


In [10]:
from langchain_core.tools import tool
@tool
def add_numbers(a:int,b:int)-> int:
  """add two numbers."""
  return a+b
print(add_numbers.invoke({"a":10,"b":5}))

15


In [11]:
llm_with_tools=llm.bind_tools([add_numbers])
print("tool connected to llm")

tool connected to llm


In [12]:
response=llm_with_tools.invoke(
    "what is 25+17? use the availale tool"
)
print(response.tool_calls)

[{'name': 'add_numbers', 'args': {'a': 25, 'b': 17}, 'id': 'fc_ef2ec7af-2d5a-4905-bb58-2bc8188d6fb4', 'type': 'tool_call'}]


In [13]:
tool_call=response.tool_calls[0]
result=add_numbers.invoke(tool_call["args"])
print("Tool:", tool_call["name"])
print("Result:", result)


Tool: add_numbers
Result: 42


In [14]:
from langchain_core.messages import ToolMessage
tool_message=ToolMessage(
    content=str(result),
    tool_call_id=tool_call["id"]
)
final_response=llm_with_tools.invoke([
    response,
    tool_message
])
print(final_response.content)

The sum of 25 and 17 is **42**.


In [15]:
from langchain.agents import create_agent
agent=create_agent(
    model=llm,
    tools=[add_numbers]
)
print("agent created!")

agent created!


In [17]:
result=agent.invoke({
    "messages":[
        {"role":"user","content":"what is 125+375?"}
    ]
})
print(result["messages"][-1].content)

125 + 375 = **500**.


In [19]:
for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 50)

HumanMessage
what is 125+375?
--------------------------------------------------
AIMessage
125 + 375 = **500**.
--------------------------------------------------


In [20]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Use the add_numbers tool to calculate 125 + 375."
        }
    ]
})

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)
    print("-" * 50)

HumanMessage
Use the add_numbers tool to calculate 125 + 375.
--------------------------------------------------
AIMessage

--------------------------------------------------
ToolMessage
500
--------------------------------------------------
AIMessage
The sum of 125 and 375 is **500**.
--------------------------------------------------


In [21]:
from langchain_core.tools import tool
@tool
def multiply_numbers(a:int,b:int)->int:
  """multiply two numbers"""
  return a*b
agent=create_agent(
    model=llm,
    tools=[add_numbers,multiply_numbers]
)
print("agent has now 2 tools")

agent has now 2 tools


In [23]:
result=agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"calculate 25 multiplied by 8 using appropriate tool."
        }
    ]
})
print(result["messages"][-1].content)

The product of 25 and 8 is **200**.


In [24]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_numbers, multiply_numbers]
)

messages = [
    {"role": "user", "content": "My name is Sugam."},
    {"role": "assistant", "content": "Nice to meet you, Sugam!"},
    {"role": "user", "content": "What is my name?"}
]

result = agent.invoke({"messages": messages})

print(result["messages"][-1].content)

Your name is Sugam.


In [25]:
agent = create_agent(
    model=llm,
    tools=[add_numbers, multiply_numbers],
    system_prompt="You are a helpful math tutor. Always explain the answer briefly."
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is 25 × 8?"}
    ]
})

print(result["messages"][-1].content)

\(25 \times 8 = 200\).

Multiplying 25 by 8 can be seen as \( (20+5) \times 8 = 20 \times 8 + 5 \times 8 = 160 + 40 = 200\).


In [27]:
from pydantic import BaseModel

class MathAnswer(BaseModel):
    operation: str
    result: int

structured_llm = llm.with_structured_output(MathAnswer)

answer = structured_llm.invoke(
    "Calculate 25 * 375"
)

print("Operation:", answer.operation)
print("Result:", answer.result)

Operation: 25 * 375
Result: 9375


In [28]:
try:
    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": "Calculate 25 multiplied by 8 using the appropriate tool."
            }
        ]
    })

    print(result["messages"][-1].content)

except Exception as e:
    print("Agent error:", e)

Multiplying 25 by 8 gives \(25 \times 8 = 200\).
